# Topic 10 - Token cost and counting

Estimate a feature's **$/request** - with the input/output split - *before* writing the
feature's code.

This notebook does the arithmetic on paper-like cells: count the input, bound the output,
apply the two-price formula, then ask which single change moves the number most.

Nothing here calls a paid API. The live-usage cell talks to the local llama-server
(`127.0.0.1:8080`) if it is running, and skips silently if it is not.

# Run the below command in cmd
```
llama-server -hf Qwen/Qwen3-4B-GGUF:Q4_K_M -ngl 0 -c 4096 --host 127.0.0.1 --port 8080
```

## 1. Counting tokens without spending money

`tiktoken` counts locally and for free. It is exact for OpenAI models and a good
approximation elsewhere (tokenizers differ, but English prose usually lands within a few
percent).

If `tiktoken` is missing we fall back to the crude "~4 characters per token" heuristic so
the rest of the notebook still runs. Never defend the heuristic number in an interview.

In [7]:
import json

try:
    import tiktoken

    _ENC = tiktoken.get_encoding("cl100k_base")

    def count_tokens(text: str) -> int:
        return len(_ENC.encode(text))

    BACKEND = "tiktoken / cl100k_base (exact for OpenAI models)"
except Exception:  # noqa: BLE001 - notebook convenience

    def count_tokens(text: str) -> int:
        return max(1, round(len(text) / 4))

    BACKEND = "heuristic ~4 chars/token (approximate!)"

print("counting backend:", BACKEND)
print("'hello world' ->", count_tokens("hello world"), "tokens")

counting backend: tiktoken / cl100k_base (exact for OpenAI models)
'hello world' -> 2 tokens


## 2. The input is bigger than the question

The user types one line. The billed input is everything you attach to it: system prompt,
tool schemas, retrieved chunks, and the whole conversation history.

Build a realistic skeleton and count each component.

In [8]:
SYSTEM_PROMPT = (
    "You are a documentation assistant. Answer only from the provided context. "
    "Cite the source ids you used. If the context does not contain the answer, say "
    "\"I don't know\" rather than guessing. Keep answers under 120 words."
)

TOOL_SCHEMA = {
    "name": "search_docs",
    "description": "Search the internal documentation index.",
    "parameters": {
        "type": "object",
        "properties": {"query": {"type": "string"}},
        "required": ["query"],
    },
}

CHUNK_SEED = (
    "Deployment checklist: run migrations, drain the queue, deploy the API, warm the caches, "
    "flip the load balancer, and watch the error rate for ten minutes. Roll back if latency "
    "doubles or the error rate exceeds one percent. "
)

HISTORY_SEED = (
    "user: how do we deploy safely?\n"
    "assistant: drain traffic, migrate, deploy, warm the caches, flip the balancer, "
    "and watch the error rate before restoring traffic.\n"
)


def repeat_until(text: str, approx_tokens: int) -> str:
    """Pad `text` with copies of itself until it reaches the target size."""
    out = ""
    while count_tokens(out) < approx_tokens:
        out += text
    return out


USER_QUESTION = "How do we deploy the API without dropping requests?"

parts = {
    "system prompt": SYSTEM_PROMPT,
    "tool schema (json)": json.dumps(TOOL_SCHEMA),
    "retrieved chunks (8 x ~400)": repeat_until(CHUNK_SEED, 400) * 8,
    "history (~5 turns)": repeat_until(HISTORY_SEED, 800),
    "user question": USER_QUESTION,
}

counts = {name: count_tokens(text) for name, text in parts.items()}
total_input = sum(counts.values())

for name, n in counts.items():
    print(f"{name:<30}{n:>8} tokens")
print(f"{'TOTAL INPUT':<30}{total_input:>8} tokens")

question_share = counts["user question"] / total_input
print(f"\nThe actual question is {question_share:.1%} of what you pay for.")

system prompt                       48 tokens
tool schema (json)                  46 tokens
retrieved chunks (8 x ~400)       3313 tokens
history (~5 turns)                 805 tokens
user question                       10 tokens
TOTAL INPUT                       4222 tokens

The actual question is 0.2% of what you pay for.


**Read that last line again.** The question is the cheapest thing in the request. Every
field you add above it is billed *on every single call*.

## 3. The two-price formula

```
cost = input_tokens  / 1e6 * input_price
     + output_tokens / 1e6 * output_price
```

Prices differ per model tier, and **output is typically several times the input price**.
The numbers below are illustrative placeholders - replace them with the provider's current
page. What is stable across providers and time is the *shape*: small tiers are roughly an
order of magnitude cheaper than frontier tiers, and output costs more than input within
every tier.

In [9]:
PRICING = {
    # USD per 1,000,000 tokens - ILLUSTRATIVE ONLY, replace with real numbers
    "small":    {"input": 0.30, "output": 1.20},
    "mid":      {"input": 3.00, "output": 15.00},
    "frontier": {"input": 15.00, "output": 60.00},
}


def cost_per_request(input_tokens: int, output_tokens: int, tier: str) -> float:
    price = PRICING[tier]
    return (
        input_tokens / 1_000_000 * price["input"]
        + output_tokens / 1_000_000 * price["output"]
    )


print("tier       input$   output$   $/request (this call)")
for tier in PRICING:
    c = cost_per_request(total_input, 350, tier)
    print(f"{tier:<10}{PRICING[tier]['input']:>8.2f}{PRICING[tier]['output']:>10.2f}{c:>14.6f}")

tier       input$   output$   $/request (this call)
small         0.30      1.20      0.001687
mid           3.00     15.00      0.017916
frontier     15.00     60.00      0.084330


## 4. Worked example - the RAG answer feature

One request: `total_input` tokens in, an estimated 350 tokens out. Then multiply by volume.

In [10]:
OUTPUT_TOKENS = 350
TIER = "mid"

per_request = cost_per_request(total_input, OUTPUT_TOKENS, TIER)
per_day = per_request * 5_000
per_month = per_day * 30

print(f"per request : ${per_request:.6f}")
print(f"per day     : ${per_day:,.2f}   (5,000 requests/day)")
print(f"per month   : ${per_month:,.2f}")

per request : $0.017916
per day     : $89.58   (5,000 requests/day)
per month   : $2,687.40


## 5. Sensitivity - what actually moves the number?

A cost estimate without a sensitivity sentence is not an estimate, it is a screenshot.
Change one lever at a time and watch the per-request price.

In [11]:
def scenario(label, input_tokens, output_tokens, tier=TIER):
    c = cost_per_request(input_tokens, output_tokens, tier)
    return label, c, c / per_request - 1


base = per_request
rows = [
    scenario("baseline", total_input, OUTPUT_TOKENS),
    scenario(
        "retrieval 8 -> 4 chunks",
        total_input - counts["retrieved chunks (8 x ~400)"] // 2,
        OUTPUT_TOKENS,
    ),
    scenario("history off (single turn)", total_input - counts["history (~5 turns)"], OUTPUT_TOKENS),
    scenario("tighter output (350 -> 150)", total_input, 150),
    # cache the static prefix (system + tool schema) at ~10% of input price
    scenario(
        "cache static prefix @10%",
        int(total_input * 0.9 + (counts["system prompt"] + counts["tool schema (json)"]) * 0.1),
        OUTPUT_TOKENS,
    ),
    scenario("answer on small tier", total_input, OUTPUT_TOKENS, tier="small"),
]

print(f"{'change':<32}{'$/request':>12}{'delta':>10}")
for label, c, delta in rows:
    print(f"{label:<32}{c:>12.6f}{delta:>9.1%}")

change                             $/request     delta
baseline                            0.017916     0.0%
retrieval 8 -> 4 chunks             0.012948   -27.7%
history off (single turn)           0.015501   -13.5%
tighter output (350 -> 150)         0.014916   -16.7%
cache static prefix @10%            0.016677    -6.9%
answer on small tier                0.001687   -90.6%


The user's 20-token question never appears in this table, because it never mattered. Cost
is dominated by *context you chose to send* and *output length you allowed*.

## 6. Reconcile against real usage (local endpoint, optional)

An estimate is only trustworthy once you have compared it with returned usage at least
once. This cell calls the local llama-server if it is up; otherwise it prints a skip note
and moves on.

In [12]:
import os

import requests

BASE_URL = os.getenv("LLM_BASE_URL", "http://127.0.0.1:8080/v1")
MODEL = os.getenv("LLM_MODEL", "Qwen/Qwen3-4B-GGUF:Q4_K_M")

SAMPLE_PROMPT = "Summarise the deployment checklist above in one sentence."


def real_usage(prompt: str, max_tokens: int = 96):
    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.0,
        "max_tokens": max_tokens,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    r = requests.post(f"{BASE_URL}/chat/completions", json=payload, timeout=300)
    r.raise_for_status()
    return r.json()


estimate = count_tokens(SAMPLE_PROMPT)
print(f"our estimate of the input : {estimate} tokens")

try:
    data = real_usage(SAMPLE_PROMPT)
    usage = data.get("usage") or {}
    actual_in = usage.get("prompt_tokens", 0)
    actual_out = usage.get("completion_tokens", 0)
    print(f"provider prompt_tokens    : {actual_in} tokens")
    print(f"provider completion_tokens: {actual_out} tokens")
    if actual_in:
        print(f"estimate error            : {actual_in - estimate} tokens")
    print("\nLog the reported usage per request from now on; it is the source of truth.")
except Exception as exc:  # noqa: BLE001 - notebook convenience
    print(f"live call failed ({type(exc).__name__}: {exc}).")
    print("Confirm the local server is running, then re-run this cell.")

our estimate of the input : 11 tokens
provider prompt_tokens    : 23 tokens
provider completion_tokens: 27 tokens
estimate error            : 12 tokens

Log the reported usage per request from now on; it is the source of truth.


## 7. Cost hygiene checklist

Fill this in before writing a new feature:

```
[ ] Model tier chosen deliberately, with a stated reason
[ ] Prompt skeleton counted (system + examples + tools)
[ ] Realistic request counted, not a best-case one
[ ] Output ceiling set (max_tokens) and its cost computed
[ ] $/request written down
[ ] x requests/day and x users -> $/day and $/month
[ ] Sensitivity named: which single change moves the number most
[ ] Retry and eval costs included
[ ] Returned usage logged per request and reconciled once
[ ] Alert threshold on median input tokens, median output tokens, daily spend
```

## Practice

1. Swap in the *real* prices for two models you can actually use and recompute the table.
   How far off was the illustrative tier ratio?
2. Add a reasoning-model scenario where 2,000 thinking tokens are billed as output. What
   does that do to $/request?
3. Model a prompt-cache *write* at 1.25x input price on the first call, then reads at 0.1x
   on the next hundred. Compute the blended cost per request across 100 calls.
4. Add a `retry_rate` of 5% and charge the retried call at full price. How much does the
   monthly figure move?
5. Write your own one-sentence cost estimate for a feature you actually want to build, in
   the form "$X per request, dominated by Y, cut by Z".